# LeetCode 259 — 3Sum Smaller bằng Python

Link đề: https://leetcode.com/problems/3sum-smaller/ · Độ khó: **Medium** (Premium) · Chủ đề: Array, Two Pointers, Binary Search, Sorting

Skeleton LeetCode:

```python
class Solution:
    def threeSumSmaller(self, nums: List[int], target: int) -> int:
        ...
```

Ghi chú: đề bài viết theo hiểu biết về bài (bài Premium), nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (ba vòng lặp trên mọi bộ chỉ số `i < j < k`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **bộ ba chỉ số** `i < j < k` sao cho `nums[i] + nums[j] + nums[k] < target` (đếm **chỉ số**, không phải đếm các giá trị khác nhau).
2. Nhận ra vì sao **sắp xếp** giúp bài này: sau khi sắp, ta có thể **đếm cả một đoạn** các bộ ba thỏa điều kiện cùng lúc thay vì kiểm tra từng bộ.
3. Đi từ **ba vòng lặp** `O(n³)` → **sắp xếp + chặt nhị phân cho phần tử thứ ba** `O(n² log n)` → **sắp xếp + hai con trỏ** `O(n²)` → **đếm theo giá trị (miền `[-100, 100]` nhỏ)** `O(n + V²)`.
4. Hiểu tại sao khi `nums[i] + nums[j] + nums[k] < target` thì **`k - j`** bộ ba `(i, j, t)` với `j < t ≤ k` đều thỏa, và vì sao ta chuyển `j` lên chứ không phải `k` xuống.
5. Tránh các lỗi kinh điển: **quên sắp xếp**, cộng `1` thay vì `k - j`, dùng `<=` thay vì `<`, đếm bộ ba theo giá trị (bỏ qua trùng lặp), và duyệt `i` sai miền.

## 2. Đề bài & ví dụ

Cho mảng `n` số nguyên `nums` và một số nguyên `target`, tìm **số bộ ba chỉ số** `i, j, k` với `0 <= i < j < k < n` thỏa điều kiện `nums[i] + nums[j] + nums[k] < target`.

**Ví dụ 1**

- Input: `nums = [-2,0,1,3], target = 2`
- Output: `2`
- Giải thích: có hai bộ ba có tổng nhỏ hơn `2`: `[-2, 0, 1]` (tổng `-1`) và `[-2, 0, 3]` (tổng `1`).

**Ví dụ 2**

- Input: `nums = [], target = 0`
- Output: `0`

**Ví dụ 3**

- Input: `nums = [0], target = 0`
- Output: `0`

**Ràng buộc:** `n == nums.length`; `0 <= n <= 3500`; `-100 <= nums[i] <= 100`; `-100 <= target <= 100`; đầu vào được tạo sao cho đáp án `<= 10^9`.

In [1]:
from typing import List
from bisect import bisect_left
from math import comb
import random
import time

examples = [
    ([-2, 0, 1, 3], 2, 2),
    ([], 0, 0),
    ([0], 0, 0),
]
for nums, target, expected in examples:
    print(f"nums={nums}, target={target} -> kỳ vọng {expected}")

nums=[-2, 0, 1, 3], target=2 -> kỳ vọng 2
nums=[], target=0 -> kỳ vọng 0
nums=[0], target=0 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Bộ ba chỉ số `i < j < k` và số cách chọn

Đề đếm các bộ ba **chỉ số** phân biệt. Hai phần tử có cùng giá trị nhưng khác chỉ số cho các bộ ba **khác nhau**. Với `n` phần tử có `C(n, 3) = n(n-1)(n-2)/6` bộ ba (ví dụ `n = 4` có `4`; `n = 3500` có khoảng `7.1·10^9`).
#### Code demo

In [2]:
nums = [-2, 0, 1, 3]
n = len(nums)
for i in range(n):
    for j in range(i + 1, n):
        for k in range(j + 1, n):
            print((i, j, k), nums[i] + nums[j] + nums[k])
print("số bộ ba:", comb(n, 3), "| C(3500, 3) =", comb(3500, 3))

(0, 1, 2) -1
(0, 1, 3) 1
(0, 2, 3) 2
(1, 2, 3) 4
số bộ ba: 4 | C(3500, 3) = 7139709500


### 3.2. Sắp xếp và tính đơn điệu của tổng

Sau `nums.sort()` ta có `nums[0] <= nums[1] <= ... <= nums[n-1]`. Với `i`, `j` cố định, `nums[i] + nums[j] + nums[k]` **không giảm khi `k` tăng**. Nên tập các `k` thỏa "tổng `< target`" là **một đoạn liên tiếp bắt đầu ngay sau `j`** — ta chỉ cần biết **điểm cuối** của đoạn đó. Thứ tự các chỉ số bị thay đổi bởi sắp xếp **không ảnh hưởng đáp án** vì tổng không phụ thuộc thứ tự và ta chỉ đếm số **bộ ba** (tập 3 phần tử phân biệt).
#### Code demo

In [3]:
nums = sorted([3, 0, -2, 1])
print(nums)
i, j = 0, 1
for k in range(j + 1, len(nums)):
    print(f"k={k}: {nums[i]} + {nums[j]} + {nums[k]} = {nums[i] + nums[j] + nums[k]}")

[-2, 0, 1, 3]
k=2: -2 + 0 + 1 = -1
k=3: -2 + 0 + 3 = 1


### 3.3. Hai con trỏ trên mảng đã sắp xếp

Với `i` cố định, đặt `j = i + 1` (đầu trái) và `k = n - 1` (đầu phải). Xét tổng `s = nums[i] + nums[j] + nums[k]`:

- nếu `s < target`: **mọi** `t` với `j < t <= k` cũng cho `nums[i] + nums[j] + nums[t] <= s < target` (vì `nums[t] <= nums[k]`) ⇒ có **`k - j`** bộ ba `(i, j, t)`; sau đó **tăng `j`** (đã dùng hết khả năng với `j` này).
- ngược lại (`s >= target`): `k` quá lớn với mọi `j' >= j` (vì `nums[j'] >= nums[j]`) ⇒ **giảm `k`**.

Mỗi bước dịch một con trỏ nên mỗi `i` tốn `O(n)`.
#### Code demo

In [4]:
nums = sorted([-2, 0, 1, 3])
target = 2
i = 0
j, k, count = i + 1, len(nums) - 1, 0
while j < k:
    s = nums[i] + nums[j] + nums[k]
    if s < target:
        print(f"j={j}, k={k}: s={s} < {target}: cộng k - j = {k - j} bộ ba, tăng j")
        count += k - j
        j += 1
    else:
        print(f"j={j}, k={k}: s={s} >= {target}: giảm k")
        k -= 1
print("i=0 đóng góp", count)

j=1, k=3: s=1 < 2: cộng k - j = 2 bộ ba, tăng j
j=2, k=3: s=2 >= 2: giảm k
i=0 đóng góp 2


### 3.4. `bisect_left`: tìm vị trí chặn trong mảng đã sắp

`bisect_left(a, x, lo, hi)` trả chỉ số đầu tiên `p` trong `a[lo:hi]` sao cho `a[p] >= x` (số phần tử **nhỏ hơn `x`** trong đoạn là `p - lo`). Chạy trong `O(log n)`. Dùng để đếm nhanh "có bao nhiêu `nums[k]` (với `k > j`) thỏa `nums[k] < limit`".
#### Code demo

In [5]:
a = [-2, 0, 1, 3]
print(bisect_left(a, 2))                 # 3: có 3 phần tử < 2 trong cả mảng
print(bisect_left(a, 2, 2, len(a)))      # 3: trong đoạn a[2:] có bisect - lo = 3 - 2 = 1 phần tử < 2
print(bisect_left(a, -5), bisect_left(a, 100))

3
3
0 4


### 3.5. Đếm theo giá trị khi miền giá trị nhỏ

Vì `nums[i] ∈ [-100, 100]` (chỉ `201` giá trị), ta có thể đếm `cnt[v]` = số lần xuất hiện của `v` rồi đếm các bộ ba **theo giá trị** `v1 <= v2 <= v3`, nhân với số cách chọn **chỉ số** tương ứng: nếu cả ba giá trị khác nhau thì `cnt[v1]·cnt[v2]·cnt[v3]`; nếu hai giá trị bằng nhau thì `C(cnt[v], 2)·cnt[khác]`; nếu cả ba bằng nhau thì `C(cnt[v], 3)`.
#### Code demo

In [6]:
from collections import Counter
cnt = Counter([1, 1, 1, 2])
print(comb(cnt[1], 3), comb(cnt[1], 2) * cnt[2])     # 1 bộ ba (1,1,1); 3 bộ ba (1,1,2)

1 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `nums = [-2, 0, 1, 3]` (đã sắp), `target = 2`. Các bộ ba (theo chỉ số):

| bộ ba | tổng | `< 2`? |
|---|---|---|
| `(0,1,2)`: `-2, 0, 1` | `-1` | có |
| `(0,1,3)`: `-2, 0, 3` | `1` | có |
| `(0,2,3)`: `-2, 1, 3` | `2` | không |
| `(1,2,3)`: `0, 1, 3` | `4` | không |

Đáp án `2`. Nhận xét: với `i = 0`, `j = 1` thì các `k = 2, 3` **cùng thỏa** (tổng `-1` và `1`) — **một đoạn liên tiếp**; với `j = 2` thì `k = 3` không thỏa; đoạn các `k` thỏa luôn là **tiền tố** của các `k > j` vì mảng đã sắp. Biết **điểm cuối** `k` của đoạn là đủ để cộng `k - j` mà không đếm từng bộ.

## 5. Cách 1 — Brute Force / baseline: ba vòng lặp lồng nhau

**Ý tưởng.** Duyệt mọi `i < j < k`, tính tổng, đếm nếu `< target`. Không cần sắp xếp.
#### Code demo

In [7]:
def three_sum_smaller_bruteforce(nums, target):
    n = len(nums)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[i] + nums[j] + nums[k] < target:
                    count += 1
    return count


for nums, target, expected in examples:
    print(nums, target, "->", three_sum_smaller_bruteforce(nums, target), "(kỳ vọng", expected, ")")

[-2, 0, 1, 3] 2 -> 2 (kỳ vọng 2 )
[] 0 -> 0 (kỳ vọng 0 )
[0] 0 -> 0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`: **Time `O(n³)`** (`C(n, 3)` bộ ba); với `n = 3500` là `~7·10^9` phép — **không thể**; **Space `O(1)`**.

**Nút thắt.** Ta kiểm tra **từng** bộ ba dù tổng đơn điệu theo `k`. Sau khi sắp xếp, các `k` thỏa tạo một đoạn nên có thể đếm bằng một phép tính.

## 6. Ý tưởng thuật toán / bước đột phá

**Sắp xếp** mảng (`O(n log n)`; không đổi đáp án vì ta đếm **bộ ba phần tử phân biệt**, độc lập thứ tự). Khi đó, với mỗi cặp `(i, j)` (`i < j`), các `k > j` thỏa `nums[k] < target - nums[i] - nums[j]` tạo **một đoạn `j+1 .. p-1`**.

- **Chặt nhị phân:** tìm `p` bằng `bisect_left` ⇒ cộng `p - (j + 1)` ⇒ `O(n² log n)`.
- **Hai con trỏ:** với mỗi `i`, đưa `j` từ trái, `k` từ phải; khi `s < target` cộng `k - j`; ⇒ `O(n²)`. Với `n = 3500`: `~1.2·10^7` phép.
- **Đếm theo giá trị:** miền giá trị nhỏ ⇒ `O(V²)` với `V = 201` (không phụ thuộc `n`).

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp + chặt nhị phân cho phần tử thứ ba

**Giải thích.** Với mỗi cặp `(i, j)`, tìm `p = bisect_left(nums, target - nums[i] - nums[j], j + 1, n)` = chỉ số đầu tiên `>= limit` trong đoạn `nums[j+1:]`; số `k` thỏa là `p - (j + 1)`.
#### Code demo

In [8]:
def three_sum_smaller_bisect(nums, target):
    a = sorted(nums)                                   # không sửa đầu vào
    n = len(a)
    count = 0
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            limit = target - a[i] - a[j]               # cần a[k] < limit
            p = bisect_left(a, limit, j + 1, n)        # vị trí đầu tiên có a[p] >= limit
            count += p - (j + 1)
    return count


for nums, target, expected in examples:
    print(nums, target, "->", three_sum_smaller_bisect(nums, target), "(kỳ vọng", expected, ")")

[-2, 0, 1, 3] 2 -> 2 (kỳ vọng 2 )
[] 0 -> 0 (kỳ vọng 0 )
[0] 0 -> 0 (kỳ vọng 0 )


**Độ phức tạp Cách 2:** **Time `O(n² log n)`** (`O(n²)` cặp, mỗi cặp một lần chặt nhị phân `O(log n)`), **Space `O(n)`** cho bản sao đã sắp.

### Cách 3 — Sắp xếp + hai con trỏ (cách nộp chuẩn)

**Giải thích.** Như mục 3.3: với mỗi `i`, hai con trỏ `j = i + 1`, `k = n - 1`; `s < target` ⇒ `ans += k - j; j += 1`; ngược lại `k -= 1`.
#### Code demo

In [9]:
def three_sum_smaller_two_pointers(nums, target):
    a = sorted(nums)
    n = len(a)
    count = 0
    for i in range(n - 2):
        j, k = i + 1, n - 1
        while j < k:
            if a[i] + a[j] + a[k] < target:
                count += k - j                         # mọi t trong (j, k] đều thỏa với i, j
                j += 1
            else:
                k -= 1
    return count


for nums, target, expected in examples:
    print(nums, target, "->", three_sum_smaller_two_pointers(nums, target), "(kỳ vọng", expected, ")")

[-2, 0, 1, 3] 2 -> 2 (kỳ vọng 2 )
[] 0 -> 0 (kỳ vọng 0 )
[0] 0 -> 0 (kỳ vọng 0 )


In [10]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def threeSumSmaller(self, nums: List[int], target: int) -> int:
        nums.sort()
        ans, n = 0, len(nums)
        for i in range(n - 2):
            j, k = i + 1, n - 1
            while j < k:
                x = nums[i] + nums[j] + nums[k]
                if x < target:
                    ans += k - j
                    j += 1
                else:
                    k -= 1
        return ans


print(Solution().threeSumSmaller([-2, 0, 1, 3], 2))   # 2

2


**Độ phức tạp Cách 3:** **Time `O(n²)`** (sắp xếp `O(n log n)` + `n` lượt hai con trỏ mỗi lượt `O(n)`), **Space `O(1)`** nếu sắp xếp tại chỗ (`nums.sort()`; `O(n)` nếu `sorted`).

### Cách 4 — Đếm theo giá trị (tận dụng miền `[-100, 100]`)

**Giải thích.** Đếm `cnt[v]` và dựng **tổng tiền tố** để biết nhanh số phần tử có giá trị trong một đoạn. Duyệt mọi cặp giá trị `v1 <= v2`:

- nếu `v1 < v2`: số bộ ba `(v1, v2, v3)` với `v3 > v2` và `v3 < target - v1 - v2` là `cnt[v1]·cnt[v2]·(số phần tử có giá trị trong (v2, giới hạn))`; cộng thêm trường hợp `v3 = v2` (khi `v1 + 2·v2 < target`): `cnt[v1]·C(cnt[v2], 2)`.
- nếu `v1 == v2`: `C(cnt[v1], 2)·(số phần tử có giá trị trong (v1, giới hạn))`, cộng thêm `C(cnt[v1], 3)` khi `3·v1 < target`.

Mỗi bộ ba theo giá trị `(v1 <= v2 <= v3)` được đếm **đúng một lần**.
#### Code demo

In [11]:
def three_sum_smaller_counting(nums, target, lo=-100, hi=100):
    width = hi - lo + 1
    cnt = [0] * width
    for v in nums:
        cnt[v - lo] += 1
    prefix = [0] * (width + 1)
    for idx in range(width):
        prefix[idx + 1] = prefix[idx] + cnt[idx]

    def count_between(a, b):                           # số phần tử có giá trị trong [a, b] (đã kẹp vào [lo, hi])
        a, b = max(a, lo), min(b, hi)
        return 0 if a > b else prefix[b - lo + 1] - prefix[a - lo]

    ans = 0
    for v1 in range(lo, hi + 1):
        c1 = cnt[v1 - lo]
        if c1 == 0:
            continue
        for v2 in range(v1, hi + 1):
            c2 = cnt[v2 - lo]
            if c2 == 0:
                continue
            limit = target - v1 - v2                   # cần v3 < limit
            if v1 < v2:
                ans += c1 * c2 * count_between(v2 + 1, limit - 1)       # v1 < v2 < v3
                if v1 + 2 * v2 < target:
                    ans += c1 * comb(c2, 2)                              # v1 < v2 == v3
            else:
                ans += comb(c1, 2) * count_between(v1 + 1, limit - 1)    # v1 == v2 < v3
                if 3 * v1 < target:
                    ans += comb(c1, 3)                                    # v1 == v2 == v3
    return ans


for nums, target, expected in examples:
    print(nums, target, "->", three_sum_smaller_counting(nums, target), "(kỳ vọng", expected, ")")

[-2, 0, 1, 3] 2 -> 2 (kỳ vọng 2 )
[] 0 -> 0 (kỳ vọng 0 )
[0] 0 -> 0 (kỳ vọng 0 )


**Độ phức tạp Cách 4:** **Time `O(n + V²)`** với `V = 201` (không phụ thuộc `n` sau bước đếm), **Space `O(V)`**. Chỉ dùng khi miền giá trị nhỏ (đúng với đề này); với giá trị lớn dùng Cách 3.

> Cách 3 là bản nộp phổ biến. Cách 2 giúp thấy "một đoạn `k` thỏa". Cách 4 minh họa tư duy tận dụng ràng buộc.

## 8. TRACE từng bước

Ta trace Cách 3 trên `[-2, 0, 1, 3]`, `target = 2` và trên ví dụ có trùng `[1, 1, 1, 2, 2]`, `target = 6`.
#### Code demo

In [12]:
def trace_two_pointers(nums, target):
    a = sorted(nums)
    n = len(a)
    print(f"sau khi sắp xếp: {a}, target={target}")
    total = 0
    for i in range(n - 2):
        j, k = i + 1, n - 1
        print(f"--- i={i} (a[i]={a[i]})")
        while j < k:
            s = a[i] + a[j] + a[k]
            if s < target:
                total += k - j
                print(f"  j={j}, k={k}: {a[i]}+{a[j]}+{a[k]}={s} < {target} -> +{k - j} (các t trong ({j},{k}]), tổng={total}, j={j + 1}")
                j += 1
            else:
                print(f"  j={j}, k={k}: {a[i]}+{a[j]}+{a[k]}={s} >= {target} -> k={k - 1}")
                k -= 1
    print("kết quả:", total)
    return total


trace_two_pointers([-2, 0, 1, 3], 2)
print()
trace_two_pointers([1, 1, 1, 2, 2], 6)

sau khi sắp xếp: [-2, 0, 1, 3], target=2
--- i=0 (a[i]=-2)
  j=1, k=3: -2+0+3=1 < 2 -> +2 (các t trong (1,3]), tổng=2, j=2
  j=2, k=3: -2+1+3=2 >= 2 -> k=2
--- i=1 (a[i]=0)
  j=2, k=3: 0+1+3=4 >= 2 -> k=2
kết quả: 2

sau khi sắp xếp: [1, 1, 1, 2, 2], target=6
--- i=0 (a[i]=1)
  j=1, k=4: 1+1+2=4 < 6 -> +3 (các t trong (1,4]), tổng=3, j=2
  j=2, k=4: 1+1+2=4 < 6 -> +2 (các t trong (2,4]), tổng=5, j=3
  j=3, k=4: 1+2+2=5 < 6 -> +1 (các t trong (3,4]), tổng=6, j=4
--- i=1 (a[i]=1)
  j=2, k=4: 1+1+2=4 < 6 -> +2 (các t trong (2,4]), tổng=8, j=3
  j=3, k=4: 1+2+2=5 < 6 -> +1 (các t trong (3,4]), tổng=9, j=4
--- i=2 (a[i]=1)
  j=3, k=4: 1+2+2=5 < 6 -> +1 (các t trong (3,4]), tổng=10, j=4
kết quả: 10


10

Bảng tóm tắt (ví dụ 1):

| `i` | `j` | `k` | tổng | quyết định | `ans` |
|---|---|---|---|---|---|
| 0 | 1 | 3 | `-2+0+3 = 1` | `< 2`: `+ (3-1) = 2`, `j=2` | 2 |
| 0 | 2 | 3 | `-2+1+3 = 2` | `>= 2`: `k=2` (dừng) | 2 |
| 1 | 2 | 3 | `0+1+3 = 4` | `>= 2`: `k=2` (dừng) | 2 |

Đáp án `2`. Chú ý: lần cộng `k - j = 2` đã tính **cả hai** bộ ba `(0,1,2)` và `(0,1,3)` mà không kiểm tra riêng `(0,1,2)`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (sắp xếp + hai con trỏ). Đúng `class Solution`, đúng `threeSumSmaller(self, nums: List[int], target: int) -> int`. (Trong notebook ta đã sao chép bản sắp xếp ở các hàm demo; bản nộp dùng `nums.sort()` tại chỗ như thường lệ trên LeetCode.)

In [13]:
class Solution:
    def threeSumSmaller(self, nums: List[int], target: int) -> int:
        nums.sort()                                   # sắp xếp tại chỗ
        ans, n = 0, len(nums)
        for i in range(n - 2):
            j, k = i + 1, n - 1
            while j < k:
                x = nums[i] + nums[j] + nums[k]
                if x < target:
                    ans += k - j                      # các t trong (j, k] đều cho tổng < target
                    j += 1
                else:
                    k -= 1
        return ans


sol = Solution()
for nums, target, expected in examples:
    assert sol.threeSumSmaller(list(nums), target) == expected
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Sắp xếp không đổi đáp án:** mỗi bộ ba chỉ số `{i, j, k}` tương ứng một tập ba phần tử; tổng không phụ thuộc thứ tự; sắp xếp chỉ đổi nhãn chỉ số nhưng giữ **đa tập giá trị** và vì ta đếm các **tập ba chỉ số** (mỗi tập đúng một lần với `i < j < k` sau khi sắp) nên số lượng giữ nguyên.
- **Đếm `k - j` đúng:** xét `i` cố định, `j < k` đang xét với `a[i] + a[j] + a[k] < target`. Với mọi `t ∈ (j, k]`: `a[t] <= a[k]` nên `a[i] + a[j] + a[t] <= a[i] + a[j] + a[k] < target`; vậy cả `k - j` bộ ba `(i, j, t)` thỏa. Mỗi bộ ba `(i, j, t)` với `j` này được đếm đúng một lần (ở thời điểm `j` có `k` hiện tại lớn nhất mà tổng `< target`).
- **Vì sao tăng `j` sau khi cộng:** mọi `(i, j, t)` với `j` hiện tại và `t ≤ k` đã được đếm; còn `t > k` chắc chắn không thỏa (đã bị loại ở các bước giảm `k` trước đó, vì `k` chỉ giảm khi `a[i] + a[j'] + a[k] >= target` với một `j' <= j` nên cũng `>=` với `j`).
- **Vì sao giảm `k` khi `s >= target`:** `k` quá lớn với `j` này, và với mọi `j' > j` (`a[j'] >= a[j]`) cũng quá lớn, nên không bỏ sót bộ ba nào.
- **Miền `i`:** `i` chạy `0..n-3` để còn chỗ cho `j < k`.
- **Cách 2:** `bisect_left` cho vị trí đầu tiên `a[p] >= limit`, nên `a[j+1..p-1] < limit`.
- **Cách 4:** mỗi tập ba chỉ số khớp đúng một bộ ba giá trị đã sắp `v1 <= v2 <= v3` và được đếm bởi công thức tổ hợp tương ứng.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `V = 201` là số giá trị khả dĩ.

| Cách | Time | Space |
|---|---|---|
| 1. Ba vòng lặp | `O(n³)` | `O(1)` |
| 2. Sắp xếp + `bisect` | `O(n² log n)` | `O(n)` |
| 3. Sắp xếp + hai con trỏ | `O(n²)` | `O(1)`–`O(n)` |
| 4. Đếm theo giá trị | `O(n + V²)` | `O(V)` |

Với `n = 3500`: Cách 3 cỡ `~1.2·10^7` phép (cỡ 1–3 giây trong Python thuần nếu viết ngây thơ; ta sẽ đo ở phần test); Cách 4 chỉ `~2·10^4` phép sau khi đếm — **nhanh nhất** nhờ ràng buộc giá trị nhỏ; Cách 1 là `~7·10^9` (bất khả thi). Không có khác biệt average/worst đáng kể (phụ thuộc dữ liệu chỉ ở mức hằng).

## 12. Bộ test

`oracle` là brute force `O(n³)` **không sắp xếp**, ba vòng lặp theo chỉ số (đúng định nghĩa đề); dùng cho `n <= 60`. Với `n` lớn, đối chiếu Cách 2, 3, 4 với nhau (và với oracle trên tập con).
#### Code demo

In [14]:
def oracle(nums, target):
    n = len(nums)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[i] + nums[j] + nums[k] < target:
                    count += 1
    return count


impls = {
    "bruteforce": three_sum_smaller_bruteforce,
    "bisect": three_sum_smaller_bisect,
    "two_pointers": three_sum_smaller_two_pointers,
    "counting": three_sum_smaller_counting,
    "submit": lambda nums, target: sol.threeSumSmaller(list(nums), target),
}

for nums, target, expected in examples:
    assert oracle(nums, target) == expected
    for key, fn in impls.items():
        assert fn(list(nums), target) == expected, (key, nums, target, fn(list(nums), target), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [15]:
edge_cases = [
    ([], 0, 0, "mảng rỗng"),
    ([1], 5, 0, "một phần tử"),
    ([1, 2], 100, 0, "hai phần tử (chưa đủ bộ ba)"),
    ([1, 2, 3], 7, 1, "đúng ba phần tử, tổng 6 < 7"),
    ([1, 2, 3], 6, 0, "tổng bằng target không tính (nghiêm ngặt)"),
    ([0, 0, 0, 0], 1, 4, "toàn 0: C(4,3) bộ ba, tổng 0 < 1"),
    ([0, 0, 0, 0], 0, 0, "toàn 0 với target 0: tổng 0 không < 0"),
    ([-100, -100, -100], -299, 1, "tổng -300 < -299: có đúng 1 bộ ba"),
    ([100, 100, 100, 100], 100, 0, "toàn 100: tổng 300 không < 100"),
    ([-100, -100, -100, -100], 100, 4, "toàn -100: mọi bộ ba tổng -300 < 100"),
    ([3, 1, 2, 5, 4], 10, 6, "chưa sắp xếp"),
    ([-2, 0, 1, 3], 2, 2, "ví dụ của đề"),
]
for nums, target, expected, label in edge_cases:
    assert oracle(nums, target) == expected, (label, oracle(nums, target), expected)
    for key, fn in impls.items():
        assert fn(list(nums), target) == expected, (key, label, fn(list(nums), target), expected)
    print("PASS", label)

PASS mảng rỗng
PASS một phần tử
PASS hai phần tử (chưa đủ bộ ba)
PASS đúng ba phần tử, tổng 6 < 7
PASS tổng bằng target không tính (nghiêm ngặt)
PASS toàn 0: C(4,3) bộ ba, tổng 0 < 1
PASS toàn 0 với target 0: tổng 0 không < 0
PASS tổng -300 < -299: có đúng 1 bộ ba
PASS toàn 100: tổng 300 không < 100
PASS toàn -100: mọi bộ ba tổng -300 < 100
PASS chưa sắp xếp
PASS ví dụ của đề


### 12.2. Tính chất và phản ví dụ

- Đáp án `0 <= ans <= C(n, 3)`.
- **Đơn điệu theo `target`:** tăng `target` thì đáp án **không giảm**.
- **Bất biến thứ tự:** hoán vị `nums` không đổi đáp án.
- **Dịch chuyển:** cộng `d` vào mọi phần tử và `3d` vào `target` không đổi đáp án.
- **Phản ví dụ** cho lỗi quên sắp xếp: hai con trỏ trên mảng chưa sắp sẽ sai.
#### Code demo

In [16]:
random.seed(259)
for _ in range(400):
    n = random.randint(0, 20)
    nums = [random.randint(-100, 100) for _ in range(n)]
    target = random.randint(-100, 100)
    ans = sol.threeSumSmaller(list(nums), target)
    assert 0 <= ans <= comb(n, 3)
    assert sol.threeSumSmaller(list(nums), target + random.randint(0, 50)) >= ans
    shuffled = nums[:]
    random.shuffle(shuffled)
    assert sol.threeSumSmaller(shuffled, target) == ans
    d = random.randint(-50, 50)
    assert sol.threeSumSmaller([x + d for x in nums], target + 3 * d) == ans
print("PASS tính chất (cận, đơn điệu theo target, hoán vị, dịch chuyển) trên 400 mảng ngẫu nhiên")


def sai_khong_sap_xep(nums, target):
    n = len(nums)
    count = 0
    for i in range(n - 2):
        j, k = i + 1, n - 1
        while j < k:
            if nums[i] + nums[j] + nums[k] < target:                # SAI: nums chưa sắp xếp
                count += k - j
                j += 1
            else:
                k -= 1
    return count


bad = [3, 1, 2, 5, 4]
print("phản ví dụ:", bad, "target 10: đúng", sol.threeSumSmaller(list(bad), 10), "| không sắp xếp:", sai_khong_sap_xep(bad, 10))
assert sai_khong_sap_xep(bad, 10) != sol.threeSumSmaller(list(bad), 10)

PASS tính chất (cận, đơn điệu theo target, hoán vị, dịch chuyển) trên 400 mảng ngẫu nhiên
phản ví dụ: [3, 1, 2, 5, 4] target 10: đúng 6 | không sắp xếp: 7


### 12.3. Đối chiếu ngẫu nhiên (có seed) và `n` lớn
#### Code demo

In [17]:
random.seed(4259)
for trial in range(1500):
    n = random.randint(0, 30)
    span = random.choice([2, 10, 100])
    nums = [random.randint(-span, span) for _ in range(n)]
    target = random.randint(-100, 100)
    truth = oracle(nums, target)
    for key, fn in impls.items():
        assert fn(list(nums), target) == truth, (key, nums, target, fn(list(nums), target), truth)
print("PASS 1500 bộ ngẫu nhiên nhỏ (nhiều giá trị trùng): mọi cách khớp oracle")

# n lớn: so Cách 2, 3, 4 (vét cạn O(n^3) không chạy nổi)
random.seed(3500)
for trial in range(3):
    n = [500, 1500, 3500][trial]
    span = [3, 100, 100][trial]
    nums = [random.randint(-span, span) for _ in range(n)]
    target = 5 if span == 3 else random.randint(-20, 20)
    ref = None
    row = []
    for key in ("two_pointers", "counting", "submit", "bisect"):
        if key == "bisect" and n > 1000:
            continue                                  # O(n^2 log n) chậm hơn; đã phủ ở n nhỏ
        t0 = time.perf_counter()
        value = impls[key](list(nums), target)
        dt = (time.perf_counter() - t0) * 1000
        ref = value if ref is None else ref
        assert value == ref, (key, n, value, ref)
        row.append(f"{key}={dt:8.1f}ms")
    print(f"n={n:>4}, span={span:>3}, target={target:>4}: ans={ref:>12} | " + " ".join(row))
print("PASS n lớn: các cách khớp nhau")

PASS 1500 bộ ngẫu nhiên nhỏ (nhiều giá trị trùng): mọi cách khớp oracle
n= 500, span=  3, target=   5: ans=    18191520 | two_pointers=     9.5ms counting=     0.1ms submit=     8.3ms bisect=    15.4ms
n=1500, span=100, target=  -7: ans=   273888825 | two_pointers=    84.5ms counting=     4.5ms submit=    84.2ms


n=3500, span=100, target= -15: ans=  3101715310 | two_pointers=   478.1ms counting=     5.3ms submit=   435.1ms
PASS n lớn: các cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên sắp xếp
Xem `sai_khong_sap_xep` ở phần test: trả sai. **Vì sao sai:** lập luận "mọi `t` trong `(j, k]` đều thỏa" cần `a[t] <= a[k]`, chỉ đúng khi mảng **đã sắp xếp**. **Code đúng:** `nums.sort()` ở đầu.

### Lỗi 2: Cộng `1` thay vì `k - j`
#### Code sai

In [18]:
def sai_cong_1(nums, target):
    a = sorted(nums)
    n = len(a)
    count = 0
    for i in range(n - 2):
        j, k = i + 1, n - 1
        while j < k:
            if a[i] + a[j] + a[k] < target:
                count += 1                                          # SAI: phải cộng k - j
                j += 1
            else:
                k -= 1
    return count


print("đúng:", sol.threeSumSmaller([-2, 0, 1, 3], 2), "| sai:", sai_cong_1([-2, 0, 1, 3], 2))
print("đúng:", sol.threeSumSmaller([1, 1, 1, 1, 1], 100), "| sai:", sai_cong_1([1, 1, 1, 1, 1], 100))

đúng: 2 | sai: 1
đúng: 10 | sai: 6


**Vì sao sai:** một lần `s < target` chứng minh cho **cả đoạn** `t ∈ (j, k]`, tức `k - j` bộ ba; cộng `1` đếm thiếu.

### Lỗi 3: Dùng `<=` thay vì `<`
#### Code sai

In [19]:
def sai_nho_hon_hoac_bang(nums, target):
    a = sorted(nums)
    n = len(a)
    count = 0
    for i in range(n - 2):
        j, k = i + 1, n - 1
        while j < k:
            if a[i] + a[j] + a[k] <= target:                         # SAI: đề yêu cầu NHỎ HƠN target
                count += k - j
                j += 1
            else:
                k -= 1
    return count


print("[1,2,3], target 6: đúng", sol.threeSumSmaller([1, 2, 3], 6), "| sai:", sai_nho_hon_hoac_bang([1, 2, 3], 6))

[1,2,3], target 6: đúng 0 | sai: 1


**Vì sao sai:** tổng `1 + 2 + 3 = 6` **không** nhỏ hơn `6`. Đề dùng so sánh **nghiêm ngặt**.

### Lỗi 4: Di chuyển nhầm con trỏ (tăng `k` hoặc không dịch con trỏ nào)
#### Code sai

In [20]:
def sai_dich_con_tro(nums, target):
    a = sorted(nums)
    n = len(a)
    count = 0
    for i in range(n - 2):
        j, k = i + 1, n - 1
        guard = 0
        while j < k and guard < 10_000:                              # guard chỉ để demo, tránh treo notebook
            guard += 1
            if a[i] + a[j] + a[k] < target:
                count += k - j
                k -= 1                                               # SAI: nên tăng j (đã đếm hết các t trong (j, k])
            else:
                k -= 1
    return count


print("đúng:", sol.threeSumSmaller([-2, 0, 1, 3, 4, -1], 3), "| sai (giảm k thay vì tăng j):", sai_dich_con_tro([-2, 0, 1, 3, 4, -1], 3))

đúng: 10 | sai (giảm k thay vì tăng j): 13


**Vì sao sai:** khi `s < target` ta đã đếm hết `(j, t)` với `t ≤ k`; giữ nguyên `j` mà giảm `k` làm **đếm lại** những bộ ba đó nhiều lần (và bỏ lỡ `j` kế tiếp). Phải **tăng `j`**.

### Lỗi 5: Duyệt `i` sai miền hoặc đếm theo giá trị (bỏ qua trùng lặp)
#### Code sai

In [21]:
def sai_dem_theo_gia_tri(nums, target):
    values = sorted(set(nums))                                       # SAI: bỏ trùng thì mất các bộ ba chỉ số khác nhau
    n = len(values)
    count = 0
    for i in range(n - 2):
        j, k = i + 1, n - 1
        while j < k:
            if values[i] + values[j] + values[k] < target:
                count += k - j
                j += 1
            else:
                k -= 1
    return count


print("[1,1,1,1], target 10: đúng", sol.threeSumSmaller([1, 1, 1, 1], 10), "| bỏ trùng:", sai_dem_theo_gia_tri([1, 1, 1, 1], 10))

[1,1,1,1], target 10: đúng 4 | bỏ trùng: 0


**Vì sao sai:** đề đếm **chỉ số**; bốn số `1` cho `C(4,3) = 4` bộ ba khác nhau, còn sau khi bỏ trùng chỉ còn một giá trị `1` ⇒ `0`. **Code đúng:** giữ nguyên mọi phần tử (chỉ sắp xếp), và `i` chạy `0..n-3`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**đếm bộ ba/bộ đôi có tổng nhỏ hơn / lớn hơn `target`**" → **sắp xếp + hai con trỏ**, và cộng **`k - j`** khi thỏa.
- Khác **3Sum** (đếm bộ ba có tổng **bằng** `0`, cần bỏ trùng): ở đây đếm **chỉ số** nên **không bỏ trùng**.
- Khi một điều kiện thỏa với **một đoạn liên tiếp** sau khi sắp, hãy **cộng cả đoạn** (hai con trỏ) hoặc tìm đầu mút bằng **chặt nhị phân**.
- Giá trị **bị chặn nhỏ** (`|x| <= 100`) → nghĩ ngay **đếm theo giá trị** (`O(V²)`), độc lập `n`.
- Dạng "k-Sum Smaller": cố định `k - 2` phần tử, hai con trỏ cho hai phần tử cuối → `O(n^(k-1))`.
- Luôn thử mảng rỗng, một/hai phần tử, tổng **bằng** `target` (không tính), toàn phần tử bằng nhau.

## 15. Biến thể của bài

1. **Two Sum Less Than K** (LeetCode 1099): tổng hai số lớn nhất nhỏ hơn `K` — hai con trỏ.
2. **3Sum** (15), **3Sum Closest** (16): bộ ba có tổng bằng / gần `target` nhất.
3. **Count Triplets with Sum Less Than Target (với nhiều truy vấn):** sắp một lần, mỗi truy vấn `O(n²)`.
4. **4Sum II** (454): đếm bộ bốn từ bốn mảng bằng bảng băm.
5. **Subarray Sum less than K** (713): hai con trỏ trượt cửa sổ.
6. Trả về **chính các bộ ba** (không chỉ đếm): lúc đó đầu ra có thể rất lớn.
#### Code demo

In [22]:
def two_sum_less_than_k(nums, k):
    """LeetCode 1099: tổng lớn nhất của hai phần tử (i < j) mà nhỏ hơn k, hoặc -1."""
    a = sorted(nums)
    i, j, best = 0, len(a) - 1, -1
    while i < j:
        s = a[i] + a[j]
        if s < k:
            best = max(best, s)
            i += 1
        else:
            j -= 1
    return best


def three_sum_smaller_triplets(nums, target):
    """Liệt kê các bộ ba chỉ số (theo mảng đã sắp) có tổng < target."""
    a = sorted(nums)
    res = []
    for i in range(len(a) - 2):
        for j in range(i + 1, len(a) - 1):
            for k in range(j + 1, len(a)):
                if a[i] + a[j] + a[k] < target:
                    res.append((a[i], a[j], a[k]))
    return res


print(two_sum_less_than_k([34, 23, 1, 24, 75, 33, 54, 8], 60))      # 58
print(three_sum_smaller_triplets([-2, 0, 1, 3], 2))                  # [(-2, 0, 1), (-2, 0, 3)]

58
[(-2, 0, 1), (-2, 0, 3)]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ba vòng lặp | thử mọi `i < j < k` | `O(n³)` | `O(1)` | oracle, `n <= 60` |
| 2. Sắp xếp + `bisect` | cố định `(i, j)`, chặt nhị phân tìm điểm cuối đoạn `k` | `O(n² log n)` | `O(n)` | rõ ý "một đoạn `k` thỏa" |
| 3. Sắp xếp + hai con trỏ | cố định `i`, `s < target` ⇒ `+k-j` và `j++` | `O(n²)` | `O(1)` | **cách nộp chuẩn** |
| 4. Đếm theo giá trị | `cnt[v]`, tổ hợp theo giá trị, tiền tố | `O(n + V²)` | `O(V)` | miền giá trị nhỏ (đề này) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `comb(5, 3)`; (b) `bisect_left([1, 3, 5, 7], 5)`; (c) `Solution().threeSumSmaller([1, 2, 3], 7)`; (d) `Solution().threeSumSmaller([1, 2, 3], 6)`.

**Bài 2 (tính bằng tay).** Tính `threeSumSmaller([3, 1, 0, -2], 4)`: sắp xếp, liệt kê mọi bộ ba và tổng, rồi đối chiếu với hai con trỏ.

**Bài 3 (điền TODO).** Hoàn thiện `smaller_todo(nums, target)` theo Cách 3.
#### Code demo

In [23]:
def smaller_todo(nums, target):
    a = sorted(nums)
    n = len(a)
    count = 0
    # TODO 1: duyệt i từ 0 tới n - 3
    # TODO 2: đặt j = i + 1, k = n - 1
    # TODO 3: while j < k: nếu a[i] + a[j] + a[k] < target -> count += k - j; j += 1
    # TODO 4: ngược lại -> k -= 1
    # TODO 5: trả count
    pass


# Tự làm xong rồi mới mở test:
# for nums, target, expected in examples:
#     assert smaller_todo(list(nums), target) == expected

**Bài 4 (tự viết lại).** Viết `smaller_bisect2(nums, target)` bằng `bisect_left` (Cách 2), **không nhìn lại** code mẫu; kiểm tra bằng `oracle` trên mảng ngẫu nhiên.

**Bài 5 (biến thể / mini challenge).** Viết `four_sum_smaller(nums, target)` đếm bộ **bốn** chỉ số có tổng `< target` bằng cách cố định hai chỉ số đầu và dùng hai con trỏ cho hai chỉ số cuối (độ phức tạp `O(n³)`); đối chiếu với vét cạn bốn vòng lặp trên `n <= 25`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `bisect_left` trả vị trí đầu tiên `>=`; tổng `1+2+3 = 6`. Bài 2: sắp xếp trước. Bài 5: giữ hai chỉ số cố định và hai con trỏ như Cách 3.
- **Mức 2 (vừa):** Bài 1: `10`, `2`, `1`, `0`. Bài 2: sắp `[-2, 0, 1, 3]`, target `4`: bộ ba `(-2,0,1)=-1`, `(-2,0,3)=1`, `(-2,1,3)=2`, `(0,1,3)=4`: có `3` bộ ba `< 4`. Bài 3: khung Cách 3.
- **Mức 3 (sâu):** Bài 2: hai con trỏ: `i=0`: `j=1,k=3` tổng `1 < 4` ⇒ `+2`, `j=2`: tổng `2 < 4` ⇒ `+1` (`k - j = 1`), `j=3` dừng ⇒ `3`; `i=1`: `j=2,k=3` tổng `4` không `< 4` ⇒ `k=2` ⇒ tổng `3`. Bài 5: `for i, for j in range(i+1, n-2)`: `lo, hi = j+1, n-1` rồi hai con trỏ.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert comb(5, 3) == 10 and bisect_left([1, 3, 5, 7], 5) == 2
assert Solution().threeSumSmaller([1, 2, 3], 7) == 1 and Solution().threeSumSmaller([1, 2, 3], 6) == 0

# Bài 2
assert Solution().threeSumSmaller([3, 1, 0, -2], 4) == 3 == oracle([3, 1, 0, -2], 4)


# Bài 3
def smaller_todo_ans(nums, target):
    a = sorted(nums)
    n = len(a)
    count = 0
    for i in range(n - 2):
        j, k = i + 1, n - 1
        while j < k:
            if a[i] + a[j] + a[k] < target:
                count += k - j
                j += 1
            else:
                k -= 1
    return count


for nums, target, expected in examples:
    assert smaller_todo_ans(list(nums), target) == expected


# Bài 4
def smaller_bisect2(nums, target):
    a = sorted(nums)
    n = len(a)
    total = 0
    for i in range(n - 2):
        for j in range(i + 1, n - 1):
            total += bisect_left(a, target - a[i] - a[j], j + 1, n) - (j + 1)
    return total


random.seed(12)
for _ in range(300):
    arr = [random.randint(-10, 10) for _ in range(random.randint(0, 25))]
    t = random.randint(-20, 20)
    assert smaller_bisect2(arr, t) == oracle(arr, t)


# Bài 5
def four_sum_smaller(nums, target):
    a = sorted(nums)
    n = len(a)
    count = 0
    for i in range(n - 3):
        for j in range(i + 1, n - 2):
            lo, hi = j + 1, n - 1
            while lo < hi:
                if a[i] + a[j] + a[lo] + a[hi] < target:
                    count += hi - lo
                    lo += 1
                else:
                    hi -= 1
    return count


def four_sum_smaller_brute(nums, target):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n) for k in range(j + 1, n) for m in range(k + 1, n)
               if nums[i] + nums[j] + nums[k] + nums[m] < target)


for _ in range(200):
    arr = [random.randint(-10, 10) for _ in range(random.randint(0, 20))]
    t = random.randint(-30, 30)
    assert four_sum_smaller(arr, t) == four_sum_smaller_brute(arr, t)
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (sắp xếp + hai con trỏ)

```python
class Solution:
    def threeSumSmaller(self, nums: List[int], target: int) -> int:
        nums.sort()
        ans, n = 0, len(nums)
        for i in range(n - 2):
            j, k = i + 1, n - 1
            while j < k:
                if nums[i] + nums[j] + nums[k] < target:
                    ans += k - j
                    j += 1
                else:
                    k -= 1
        return ans
```

**Checklist**

- [ ] **Sắp xếp** trước khi dùng hai con trỏ?
- [ ] Khi `tổng < target`: cộng **`k - j`** rồi **tăng `j`**; ngược lại **giảm `k`**?
- [ ] So sánh **nghiêm ngặt** `<`, và đếm **chỉ số** (không bỏ trùng giá trị)?
- [ ] `i` chạy `0..n-3`; xử lý `n < 3` (kết quả `0`)?
- [ ] Hiểu vì sao `k - j` bộ ba `(i, j, t)` với `j < t <= k` đều thỏa?
- [ ] Đã thử `[]`, `[0]`, `[1,2,3]` với target `6` và `7`, toàn phần tử bằng nhau?
- [ ] Đã đối chiếu với brute force ba vòng lặp trên mảng ngẫu nhiên nhỏ?
- [ ] Skeleton đúng `class Solution` / `def threeSumSmaller(self, nums: List[int], target: int) -> int`?